# Original dataset [AI config dataset](https://se-uhd.de/ai-config-dataset/)

In [1]:
import re
import pandas as pd
import numpy as np
from helper import load_jsonl_to_df

In [2]:
# Original dataset
configured_repos = pd.read_csv("../data/configured_repos.csv")
configured_repos = configured_repos.rename(columns={'repo_name': 'repo'})
context_df = pd.read_csv("../data/context_files.csv")
skills = pd.read_csv("../data/skills.csv")
subagents = pd.read_csv("../data/subagents.csv") # subagent definitions
commands = pd.read_csv("../data/commands.csv") # custom commands
rules = pd.read_csv("../data/rules.csv") # rule files
settings = pd.read_csv("../data/settings.csv") # settings files
hooks = pd.read_csv("../data/hooks.csv") # hook configurations
mcp = pd.read_csv("../data/mcp.csv") # MCP configurations

# Our data
pr_df = pd.read_csv("../data/pull_request_combined.csv")
pr_five_df = pd.read_csv("../data/repo_agent_metrics_comparison_mode_1.csv")
pr_feat_df = pd.read_csv("../data/prs_features.csv")
repo_df = pd.read_csv("../data/repository_combined.csv")
repo_df = repo_df.rename(columns={'id': 'repo_id'})

pr_df = pr_df.merge(
    repo_df[['repo_id', 'full_name']],
    on='repo_id',
    how='left'
).rename(columns={'full_name': 'repo'})

# Add the number of agent-authored PRs to the original repo dataframe
configured_repos = configured_repos.merge(
    pr_feat_df.groupby("repo").size().reset_index(name="agent_prs"),
    on="repo", how='left'
)

# 1. Convert string columns to datetime
# Using utc=True is safer if your timestamps include timezone offsets (like 'Z')
configured_repos['createdAt'] = pd.to_datetime(configured_repos['createdAt'], utc=True)
configured_repos['scanned_at'] = pd.to_datetime(configured_repos['scanned_at'], utc=True)

# 2. Compute the difference (Age)
# This creates a Timedelta column
configured_repos['age_delta'] = configured_repos['scanned_at'] - configured_repos['createdAt']

# 3. Extract specific units
# Age in days (as integer)
configured_repos['repo_age_days'] = configured_repos['age_delta'].dt.days

In [3]:
# print(configured_repos.iloc[0, 7])
pr_feat_df['state'].value_counts(normalize=True)

state
merged    0.7724
closed    0.2276
Name: proportion, dtype: float64

#### The projects with no agent-authored PRs

In [4]:
def generate_repo_stats_table(df: pd.DataFrame):
    """
    Generate a LaTeX table with repository statistics from GitHub PR data.
'', ''
    Args:
        df (pd.DataFrame): DataFrame containing GitHub PR data with repository metadata
        output_file (str): Path to output LaTeX file
    """
    # Step 1: Remove duplicate repositories (keep first occurrence)
    df_unique = df.sort_values("stargazers", ascending=0).drop_duplicates(
        subset=['repo'], keep='first').copy()

    # Count security PRs per repository
    # agentic_prs = df.groupby('repo').size()
    # df_unique['agentic_pr_count'] = df_unique['repo'].map(agentic_prs).fillna(0)

    # Step 3: Prepare the statistics table
    metrics = {
        'Age (days)': 'repo_age_days',
        'Stars': 'stargazers',
        'Commits': 'commits',
        'Agentic-PRs': 'agent_prs',
        'Total PRs': 'totalPullRequests',
        'Contributors': 'contributors',
        # 'Users': 'mentionableUsersCount'
    }

    # Calculate statistics for each metric
    stats_data = []
    for metric_name, col_name in metrics.items():
        values = df_unique[col_name]
        stats_data.append({
            'Metric': metric_name,
            'Min.': np.min(values),
            'Median': np.median(values),
            'Mean': np.mean(values),
            'Max.': np.max(values)
        })

    # Create stats DataFrame
    stats_df = pd.DataFrame(stats_data)
    stats_df['Max.'] = stats_df['Max.'].apply(lambda x: f"{x:,}")
    stats_df['Median'] = stats_df['Median'].apply(lambda x: f"{x:,.2f}")
    stats_df['Mean'] = stats_df['Mean'].apply(lambda x: f"{x:,.2f}")

    # Step 4: Generate LaTeX table
    latex_table = stats_df.to_latex(
        index=False,
        column_format='lrrrr',
        float_format="{:0.2f}".format,
        caption=f'Statistics of the {df['repo'].nunique():,} studied projects and {df['agent_prs'].sum():,} agentic PRs.',
        label='tab:repo-stats',
        position='t!'
    )
    latex_table = latex_table.replace(".00", "").replace(".0 ", " ")
    # Save to file
    # with open(output_file, 'w') as f:
    #     f.write(latex_table)

    return latex_table, df_unique

#### Stats of the project with at least 5 agent-authored and 5 after the creation of the first instruction file

In [9]:
COUNT_PR_THRESHOLD = 20

five_pr_mask = (pr_five_df['pr_count_after']>=COUNT_PR_THRESHOLD)
repos_with_5_pr_bef_aft = pr_five_df.loc[
    five_pr_mask,
    'repo'
].unique()
# repos_with_5_pr_bef_aft

In [11]:
len(repos_with_5_pr_bef_aft)

523

In [14]:
context_df[context_df['repo_name'].isin(repos_with_5_pr_bef_aft)].to_csv("../data/context_files_subset.csv", index=None)

In [14]:
configured_repos[configured_repos['repo'].isin(repos_with_5_pr_bef_aft)]['agent_prs'].sum()

np.float64(53153.0)

In [9]:
latex_output, repo_stat_df = generate_repo_stats_table(configured_repos[configured_repos['repo'].isin(repos_with_5_pr_bef_aft)])

In [10]:
print(latex_output)

\begin{table}[t!]
\caption{Statistics of the 188 studied projects and 18,008 agentic PRs.}
\label{tab:repo-stats}
\begin{tabular}{lrrrr}
\toprule
Metric & Min. & Median & Mean & Max. \\
\midrule
Age (days) & 571 & 1,921.50 & 2,195.14 & 6,357 \\
Stars & 10 & 2,303.50 & 10,122.20 & 145,711 \\
Commits & 375 & 4,376 & 8,902.71 & 142,284 \\
Agentic-PRs & 10 & 44 & 95.79 & 1,447 \\
Total PRs & 45 & 2,320.50 & 5,543.40 & 73,940 \\
Contributors & 5 & 87.50 & 144.38 & 463 \\
\bottomrule
\end{tabular}
\end{table}



#### Stats for projects with at least X agent-authored PR after the creation of context files

In [11]:
# Number of configured projects
num_configured_pro = configured_repos['repo'].unique()
num_pro_with_prs = pr_df['repo'].unique()
num_agentic_prs = pr_feat_df['repo'].isin(repos_with_5_pr_bef_aft).sum()

# Number of projects that have at least one agent-authored PRs (up to 2026-03-03)
print("Percentage of projects that have at least one agent-authored PR:", round(100 * (len(num_pro_with_prs) / len(num_configured_pro)), 2), f"({len(num_pro_with_prs)} out of {len(num_configured_pro)})")
print(f"The number of agent-authored PRs associated with the {len(repos_with_5_pr_bef_aft)} projects: {num_agentic_prs}")
print(f"The distribution of PRs for the {len(num_pro_with_prs)} with at least one agent-authored PR:")

num_proj_with_five_prs_bef_aft = pr_five_df.loc[
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD)&
    (pr_five_df['pr_count_after']>=COUNT_PR_THRESHOLD),
    'repo'
].unique()

# How many projects with at least COUNT_PR_THRESHOLD agent-authored PRs and COUNT_PR_THRESHOLD after have skills?
num_proj_with_skills = pr_five_df.loc[
    (pr_five_df['pr_count_after']>=COUNT_PR_THRESHOLD) &
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD) &
    (pr_five_df['repo'].isin(skills['repo_name'].values)),
    'repo'
].unique()

# How many projects with at least COUNT_PR_THRESHOLD agent-authored PRs and COUNT_PR_THRESHOLD after have sub-agents
num_proj_with_subagents = pr_five_df.loc[
    (pr_five_df['pr_count_after']>=COUNT_PR_THRESHOLD) &
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD) &
    (pr_five_df['repo'].isin(subagents['repo_name'].values)),
    'repo'
].unique()

# How many projects with at least COUNT_PR_THRESHOLD agent-authored PRs and COUNT_PR_THRESHOLD after have commands
num_proj_with_commands = pr_five_df.loc[
    (pr_five_df['pr_count_after']>=COUNT_PR_THRESHOLD) &
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD) &
    (pr_five_df['repo'].isin(commands['repo_name'].values)),
    'repo'
].unique()

# How many projects with at least COUNT_PR_THRESHOLD agent-authored PRs and COUNT_PR_THRESHOLD after have hooks
num_proj_with_hooks = pr_five_df.loc[
    (pr_five_df['pr_count_after']>=COUNT_PR_THRESHOLD) &
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD) &
    (pr_five_df['repo'].isin(hooks['repo_name'].values)),
    'repo'
].unique()

# How many projects with at least COUNT_PR_THRESHOLD agent-authored PRs and COUNT_PR_THRESHOLD after have mcp
num_proj_with_mcp = pr_five_df.loc[
    (pr_five_df['pr_count_after']>=COUNT_PR_THRESHOLD) &
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD) &
    (pr_five_df['repo'].isin(mcp['repo_name'].values)),
    'repo'
].unique()

# How many projects with at least COUNT_PR_THRESHOLD agent-authored PRs and COUNT_PR_THRESHOLD after have settings
num_proj_with_settings = pr_five_df.loc[
    (pr_five_df['pr_count_after']>=COUNT_PR_THRESHOLD) &
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD) &
    (pr_five_df['repo'].isin(settings['repo_name'].values)),
    'repo'
].unique()

print(f"Percentage of projects that have at least {COUNT_PR_THRESHOLD} agent-authored PRs before and {COUNT_PR_THRESHOLD} after the creation of context files:", round(100 * (len(num_proj_with_five_prs_bef_aft) / len(num_configured_pro)), 2), f"({len(num_proj_with_five_prs_bef_aft)} out of {len(num_configured_pro)})")
print("The number of project with at least five PRs before and five after the creation of context files:")
print(f"{' '*8} 1. Skills: {round(100 * len(num_proj_with_skills) / len(num_proj_with_five_prs_bef_aft), 2)}")
print(f"{' '*8} 2. Sub-agents: {round(100 * len(num_proj_with_subagents) / len(num_proj_with_five_prs_bef_aft), 2)}")
print(f"{' '*8} 3. Commands: {round(100 * len(num_proj_with_commands) / len(num_proj_with_five_prs_bef_aft), 2)}")
print(f"{' '*8} 4. Hooks: {round(100 * len(num_proj_with_hooks) / len(num_proj_with_five_prs_bef_aft), 2)}")
print(f"{' '*8} 5. Settings: {round(100 * len(num_proj_with_settings) / len(num_proj_with_five_prs_bef_aft), 2)}")
print(f"{' '*8} 6. MCP: {round(100 * len(num_proj_with_mcp) / len(num_proj_with_five_prs_bef_aft), 2)}")

Percentage of projects that have at least one agent-authored PR: 47.66 (2258 out of 4738)
The number of agent-authored PRs associated with the 188 projects: 18008
The distribution of PRs for the 2258 with at least one agent-authored PR:
Percentage of projects that have at least 5 agent-authored PRs before and 5 after the creation of context files: 17.88 (847 out of 4738)
The number of project with at least five PRs before and five after the creation of context files:
         1. Skills: 19.72
         2. Sub-agents: 12.16
         3. Commands: 11.57
         4. Hooks: 5.31
         5. Settings: 16.88
         6. MCP: 4.13


#### Stats for projects with at least X agent-authored PR after the creation of context files

In [14]:
# How many projects with at least COUNT_PR_THRESHOLD agent-authored PRs and COUNT_PR_THRESHOLD after have skills?
num_proj_with_skills = pr_five_df.loc[
    (pr_five_df['pr_count_after']<COUNT_PR_THRESHOLD) &
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD) &
    (pr_five_df['repo'].isin(skills['repo_name'].values)),
    'repo'
].nunique()

# How many projects with at least COUNT_PR_THRESHOLD agent-authored PRs and COUNT_PR_THRESHOLD after have sub-agents
num_proj_with_subagents = pr_five_df.loc[
    (pr_five_df['pr_count_after']<COUNT_PR_THRESHOLD) &
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD) &
    (pr_five_df['repo'].isin(subagents['repo_name'].values)),
    'repo'
].nunique()

# How many projects with at least COUNT_PR_THRESHOLD agent-authored PRs and COUNT_PR_THRESHOLD after have commands
num_proj_with_commands = pr_five_df.loc[
    (pr_five_df['pr_count_after']<COUNT_PR_THRESHOLD) &
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD) &
    (pr_five_df['repo'].isin(commands['repo_name'].values)),
    'repo'
].nunique()

# How many projects with at least COUNT_PR_THRESHOLD agent-authored PRs and COUNT_PR_THRESHOLD after have hooks
num_proj_with_hooks = pr_five_df.loc[
    (pr_five_df['pr_count_after']<COUNT_PR_THRESHOLD) &
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD) &
    (pr_five_df['repo'].isin(hooks['repo_name'].values)),
    'repo'
].nunique()

# How many projects with at least COUNT_PR_THRESHOLD agent-authored PRs and COUNT_PR_THRESHOLD after have mcp
num_proj_with_mcp = pr_five_df.loc[
    (pr_five_df['pr_count_after']<COUNT_PR_THRESHOLD) &
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD) &
    (pr_five_df['repo'].isin(mcp['repo_name'].values)),
    'repo'
].nunique()

# How many projects with at least COUNT_PR_THRESHOLD agent-authored PRs and COUNT_PR_THRESHOLD after have settings
num_proj_with_settings = pr_five_df.loc[
    (pr_five_df['pr_count_after']<COUNT_PR_THRESHOLD) &
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD) &
    (pr_five_df['repo'].isin(settings['repo_name'].values)),
    'repo'
].nunique()

print(f"The number of project with at least {COUNT_PR_THRESHOLD} after the creation of context files:")
print(f"{' '*8} 1. Skills: {round(100 * num_proj_with_skills / len(num_proj_with_five_prs_bef_aft), 2)}")
print(f"{' '*8} 2. Sub-agents: {round(100 * num_proj_with_subagents / len(num_proj_with_five_prs_bef_aft), 2)}")
print(f"{' '*8} 3. Commands: {round(100 * num_proj_with_commands / len(num_proj_with_five_prs_bef_aft), 2)}")
print(f"{' '*8} 4. Hooks: {round(100 * num_proj_with_hooks / len(num_proj_with_five_prs_bef_aft), 2)}")
print(f"{' '*8} 5. Settings: {round(100 * num_proj_with_settings / len(num_proj_with_five_prs_bef_aft), 2)}")
print(f"{' '*8} 6. MCP: {round(100 * num_proj_with_mcp / len(num_proj_with_five_prs_bef_aft), 2)}")

The number of project with at least 5 after the creation of context files:
         1. Skills: 12.28
         2. Sub-agents: 5.55
         3. Commands: 5.31
         4. Hooks: 2.13
         5. Settings: 7.79
         6. MCP: 3.31


#### The comparison among projects have have skills vs. those that do not

##### Skills

In [15]:
repos_with_skills = skills['repo_name'].unique()

skills_df = pr_five_df[
    five_pr_mask &
    pr_five_df['repo'].isin(repos_with_skills)
]

non_skills_df = pr_five_df[
    five_pr_mask &
    (~pr_five_df['repo'].isin(repos_with_skills))
]

# print("SKILLS stats among projects that have skills vs. those that do not:")
# for metric in ['closure_time', 'num_files', 'code_churn', 'num_commits', 'comments', 'description_length', 'num_inline_comments']:
#     skill_val = skills_df['metric'].median()
#     non_skill_val = non_skills_df[metric].median()
    
#     print(f"Metric: {metric}, with skills: {skill_val}, without skills: {non_skill_val}")
non_skills_df['merge_impro_pct'].describe()

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


count    151.000
mean         inf
std          NaN
min     -100.000
25%      -15.315
50%        0.000
75%       21.190
max          inf
Name: merge_impro_pct, dtype: float64

In [15]:
pr_five_df

,repo,num_context_files,mode,first_commit_context_ai_created,merge_impro_pct,merge_rate_before,merge_time_before,closure_time_before,num_files_before,code_churn_before,...,changes_requested_delta,changes_requested_eff_size,word_count,content_length,H1,H2,H3,H4,H5,H6
0,JetBrains/intellij-community,2,1,False,0.0,0.0,0.000000,0.000000,0.0,0.0,...,0.0,NaN,840,6417,1,7,6,0,0,0
1,rust-lang/rust,1,1,False,0.0,0.0,0.000000,0.777361,5.5,81.5,...,0.0,NaN,248,1797,0,4,1,0,0,0
2,dimagi/commcare-hq,1,1,False,inf,0.0,0.000000,0.000000,0.0,0.0,...,0.0,NaN,635,4438,13,5,8,0,0,0
3,mantidproject/mantid,2,1,True,30.0,0.5,264.378889,134.303472,11.0,484.0,...,0.0,negligible,1031,8500,11,7,10,0,0,0
4,microsoft/vscode,18,1,False,inf,0.0,0.000000,0.000000,0.0,0.0,...,0.0,NaN,8298,64598,14,53,49,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1656,DamageLabs/rims,1,1,True,inf,0.0,0.000000,0.000000,0.0,0.0,...,0.0,NaN,458,4414,1,4,6,0,0,0
1657,Amm1rr/WebAI-to-API,1,1,False,0.0,0.0,0.000000,0.000000,0.0,0.0,...,0.0,NaN,656,5394,6,4,11,4,0,0
1658,pydantic/monty,1,1,False,inf,0.0,0.000000,0.000000,0.0,0.0,...,0.0,NaN,3769,25765,20,12,18,6,0,0
1659,bretterer/home-assistant-rivian,1,1,False,0.0,0.0,0.000000,21.806111,5.0,203.0,...,0.0,NaN,429,3548,1,4,7,0,0,0


In [19]:
pr_five_df.loc[
    (pr_five_df['pr_count_after']<COUNT_PR_THRESHOLD) &
    (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD) &
    (pr_five_df['repo'].isin(skills['repo_name'].values)),
    'repo'
]#.nunique()

61                       artsy/eigen
138             microsoft/playwright
184               ProvableHQ/snarkVM
261                    near/nearcore
787               mountain-loop/yaak
920           ansys/pyansys-geometry
992        microsoft/vscode-cosmosdb
1337               allenai/OLMo-core
1505    wysaid/android-gpuimage-plus
1544                   wei/socialify
Name: repo, dtype: object

##### Subagents

In [16]:
repos_with_subagents = subagents['repo_name'].unique()

subagent_df = pr_five_df[
    five_pr_mask &
    pr_five_df['repo'].isin(repos_with_subagents)
]

non_subagent_df = pr_five_df[
    five_pr_mask &
    (~pr_five_df['repo'].isin(repos_with_subagents))
]

In [17]:
subagent_df['merge_impro_pct'].describe()

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


count    24.00
mean       inf
std        NaN
min     -38.59
25%      -2.37
50%       9.28
75%      53.77
max        inf
Name: merge_impro_pct, dtype: float64

##### Commands

In [18]:
repos_with_commands = rules['repo_name'].unique()

command_df = pr_five_df[
    five_pr_mask &
    pr_five_df['repo'].isin(repos_with_commands)
]

non_command_df = pr_five_df[
    five_pr_mask &
    (~pr_five_df['repo'].isin(repos_with_commands))
]

In [19]:
pr_five_df[five_pr_mask].sort_values("merge_impro_pct")[:10][['repo', 'merge_impro_pct', 'pr_count_before', 'pr_count_after']]

,repo,merge_impro_pct,pr_count_before,pr_count_after
317,kubernetes-sigs/headlamp,-100.00,12.0,5.0
603,danny-avila/LibreChat,-88.57,16.0,20.0
1086,stellar/stellar-cli,-76.19,5.0,7.0
161,risingwavelabs/risingwave,-72.73,6.0,22.0
546,runatlantis/atlantis,-62.96,10.0,9.0
736,envoyproxy/toolshed,-59.90,40.0,171.0
528,semaphoreui/semaphore,-51.92,5.0,26.0
190,apache/datafusion,-51.32,40.0,17.0
604,valkey-io/valkey-glide,-48.15,56.0,12.0
1126,lmnr-ai/lmnr,-47.39,11.0,115.0


#### 

In [20]:
first_ai_gen_df = pr_five_df[
    five_pr_mask &
    (pr_five_df['first_commit_context_ai_created'] == True)
    # pr_five_df['repo'].isin(repos_with_subagents)
]

non_first_ai_gen_df = pr_five_df[
    five_pr_mask &
    (pr_five_df['first_commit_context_ai_created'] == False)
    # (~pr_five_df['repo'].isin(repos_with_subagents))
]

In [21]:
first_ai_gen_df['word_count'].describe(), non_first_ai_gen_df['word_count'].describe()

(count       82.000000
 mean      2761.500000
 std       5739.532532
 min         77.000000
 25%        616.500000
 50%       1077.500000
 75%       2367.750000
 max      42840.000000
 Name: word_count, dtype: float64,
 count      106.000000
 mean      2142.047170
 std       2819.829638
 min         13.000000
 25%        570.250000
 50%       1170.500000
 75%       2623.000000
 max      15787.000000
 Name: word_count, dtype: float64)

#### Split the data into project context files for projects that increase the merge rate by at least 20% and those that decrease it.

In [22]:
pro_with_20_merge_rate_impro = pr_five_df.loc[
    five_pr_mask &
    (pr_five_df['merge_impro_pct']>=20),
    'repo'
].unique()
pro_with_minus_20_merge_rate_impro = pr_five_df.loc[
    five_pr_mask &
    (pr_five_df['merge_impro_pct']<=-20),
    'repo'
].unique()

In [27]:
context_df[(context_df['is_reference']==False)&(context_df['repo_name'].isin(pro_with_20_merge_rate_impro))].to_csv("../data/plus_twenty_merge_rate_context_files.csv", index=None)
context_df[(context_df['is_reference']==False)&(context_df['repo_name'].isin(pro_with_minus_20_merge_rate_impro))].to_csv("../data/minus_twenty_merge_rate_context_files.csv", index=None)

#### Print out the results of the two groups of projects regarding the qualitty of instruction files

In [34]:
minus_twenty_merge_rate_inst_df = pd.read_csv("../data/minus_twenty_merge_rate_context_complexity_metrics.csv")
plus_twenty_merge_rate_inst_df = pd.read_csv("../data/plus_twenty_merge_rate_context_complexity_metrics.csv")

In [37]:
for idx, row in plus_twenty_merge_rate_inst_df.iterrows():
    print(f"Repository: {row['repo_name']} -- Instruction file name: {row['context_file']}")
    print(f"Complexity summary: {row['complexity_summary']}")
    print("*"*50)

Repository: Automattic§studio -- Instruction file name: AGENTS.md
Complexity summary: This instruction file provides a comprehensive set of guidelines and constraints to manage the complexity of the project, emphasizing critical operations and architectural details. It enforces strict rules on coding practices and project structure, ensuring that the autonomous agent adheres to best practices and avoids common pitfalls.
**************************************************
Repository: Azure-Samples§azure-search-openai-demo -- Instruction file name: .github/instructions/bicep.instructions.md
Complexity summary: This instruction file provides a comprehensive set of best practices for writing Bicep code, emphasizing what developers should avoid to maintain code quality. It employs strict guidelines and negative constraints to guide the agent's behavior, ensuring adherence to best practices in infrastructure as code.
**************************************************
Repository: Azure-Samples

#### Exploring how dependencies are managed with coding agents

In [23]:
pr_commits_df = pd.read_csv("../data/pr_commits_combined.csv")
pr_commit_details_df = pd.read_csv("../data/pr_commit_details_combined.csv")

pr_df = pr_df.rename(columns={'id': 'pr_id'})

pr_commit_details_df = pr_commit_details_df.merge(
    pr_df[['pr_id', 'repo', 'number']],
    on='pr_id',
    how='left'
)

In [25]:
# 1. Define the files in a list for better readability
target_files = [
    "package.json", "requirements.txt", "pyproject.toml", "Pipfile",
    "pom.xml", "build.gradle", "build.gradle.kts", "Cargo.toml",
    "composer.json", "go.mod", "Gemfile", "Package.swift", "pubspec.yaml"
]

# 2. Escape dots and join with OR (|)
# The (?:^|.*/) part allows the regex to match files at the root OR inside folders
regex_pattern = r"(?:^|/)(?:" + "|".join([re.escape(f) for f in target_files]) + r")$"

# 3. Apply filtering
filtered_package_df = pr_commit_details_df[
    pr_commit_details_df["filename"].str.contains(regex_pattern, regex=True, na=False)
    & pr_commit_details_df["repo"].isin(repos_with_5_pr_bef_aft)
]

In [26]:
# filtered_package_df.groupby(["repo"]).count().reset_index(level=1).sort_values("sha")['sha']#.reset_index(names="count")
filtered_package_df.groupby(["repo", "pr_id"]).count().reset_index(level=1).sort_values("sha")['sha']#.reset_index(names="count")

repo
microsoft/vscode-cmake-tools      1
enonic/xp                         1
enkan/enkan                       1
matin/garth                       1
mlflow/mlflow                     1
                               ... 
getsentry/sentry-javascript     204
swc-project/swc                 207
skypilot-org/skypilot           247
Maintainerr/Maintainerr         286
getsentry/sentry-javascript     348
Name: sha, Length: 3174, dtype: int64

In [27]:
filtered_package_df.columns

Index(['sha', 'pr_id', 'author', 'committer', 'message', 'commit_stats_total',
       'commit_stats_additions', 'commit_stats_deletions', 'filename',
       'status', 'additions', 'deletions', 'changes', 'patch', 'repo',
       'number'],
      dtype='object')

In [29]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

# 1. Configuration
OUTPUT_DIR = 'figs'
PREFIX = 'agent_deps_'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. Dependency File Selection
dep_files = [
    "package.json", "requirements.txt", "pyproject.toml", "Pipfile",
    "pom.xml", "build.gradle", "build.gradle.kts", "Cargo.toml",
    "composer.json", "go.mod", "Gemfile", "Package.swift", "pubspec.yaml"
]
# Regex to match filenames at root or in subdirectories
regex_pattern = r"(?:^|/)(" + "|".join([re.escape(f) for f in dep_files]) + r")$"

# 3. Filtering
# Assuming your dataframe is named 'df'
dep_df = filtered_package_df[filtered_package_df["filename"].str.contains(regex_pattern, regex=True, na=False)].copy()

def save_plot(fig, name):
    filename = f"{PREFIX}{name}.pdf"
    path = os.path.join(OUTPUT_DIR, filename)
    fig.savefig(path, format='pdf', bbox_inches='tight')
    plt.close(fig)
    print(f"Saved: {path}")

# Set Visual Style
# sns.set_theme(style="whitegrid")

# # --- Plot 1: Frequency ---
# fig, ax = plt.subplots(figsize=(10, 6))
# sns.countplot(data=dep_df, y="filename", palette="viridis", 
#               order=dep_df["filename"].value_counts().index)
# ax.set_title("Frequency of Changes per Dependency File Type")
# save_plot(fig, "frequency")

# # --- Plot 2: Churn (Additions vs Deletions) ---
# fig, ax = plt.subplots(figsize=(10, 6))
# sns.scatterplot(data=dep_df, x="additions", y="deletions", hue="filename")
# ax.set_title("Dependency Churn: Additions vs. Deletions")
# save_plot(fig, "churn_scatter")

# # --- Plot 3: Impact Distribution ---
# fig, ax = plt.subplots(figsize=(10, 6))
# sns.boxplot(data=dep_df, x="changes", y="filename", showfliers=False)
# ax.set_title("Distribution of Total Changes (Lines) per File")
# save_plot(fig, "impact_distribution")

# # --- Plot 4: Top Repositories ---
# fig, ax = plt.subplots(figsize=(10, 6))
# top_repos = dep_df["repo"].value_counts().nlargest(10)
# sns.barplot(x=top_repos.values, y=top_repos.index, palette="magma")
# ax.set_title("Top 10 Repositories by Agent Dependency Activity")
# save_plot(fig, "top_repos")

# # --- Plot 5: Net Impact (Growth vs Reduction) ---
# dep_df['net_change'] = dep_df['additions'] - dep_df['deletions']
# fig, ax = plt.subplots(figsize=(10, 6))
# sns.violinplot(data=dep_df, x="net_change", y="filename", inner="quart")
# ax.set_title("Net Change (Additions - Deletions) by File")
# save_plot(fig, "net_impact")

# # --- Plot 6: Message Keywords ---
# keywords = ['update', 'bump', 'fix', 'security', 'cve', 'version']
# keyword_counts = {kw: dep_df['message'].str.contains(kw, case=False).sum() for kw in keywords}
# fig, ax = plt.subplots(figsize=(10, 6))
# ax.bar(keyword_counts.keys(), keyword_counts.values(), color='teal')
# ax.set_title("Keywords Found in Agent Dependency Commits")
# save_plot(fig, "keywords")

/var/folders/p4/wn9d12jj6g78td0tl7gx7xlw0000gp/T/ipykernel_94428/1955125879.py:23: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  dep_df = filtered_package_df[filtered_package_df["filename"].str.contains(regex_pattern, regex=True, na=False)].copy()


In [33]:
regex_pattern = r"(?:^|/)(" + "|".join([re.escape(f) for f in dep_files]) + r")$"

# Create the filtered dependency dataframe
dep_df = pr_commit_details_df[
    pr_commit_details_df["filename"].str.contains(regex_pattern, regex=True, na=False)
].copy()

# --- 2. Table 1: Repository Activity Summary ---

# Calculate Commits and unique PRs per repo
repo_summary = dep_df.groupby('repo').agg(
    total_commits=('sha', 'nunique'),
    unique_prs=('pr_id', 'nunique')
).sort_values(by='total_commits', ascending=False)

# Calculate global PR distribution stats
commit_stats = repo_summary['total_commits'].agg(['min', 'median', 'max']).to_frame().T
pr_stats = repo_summary['unique_prs'].agg(['min', 'median', 'max']).to_frame().T

# Display Table 1
# print("### Table 1: Repository Activity Summary")
# print(repo_summary)
print("### Table 1: Commit Distribution Stats")
print(commit_stats)
print("\n### Global PR Distribution Stats")
print(pr_stats)

# --- 3. Table 2: Dependency File PR Frequency ---

# Calculate unique PRs per file type
# Note: We extract the basename (e.g., 'src/package.json' -> 'package.json') for clean grouping
dep_df['file_basename'] = dep_df['filename'].str.split('/').str[-1]

file_summary = dep_df.groupby('file_basename').agg(
    unique_prs=('pr_id', 'nunique')
).sort_values(by='unique_prs', ascending=False)

# Display Table 2
print("\n### Table 2: Dependency File PR Frequency")
print(file_summary)

/var/folders/p4/wn9d12jj6g78td0tl7gx7xlw0000gp/T/ipykernel_94428/973480965.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  pr_commit_details_df["filename"].str.contains(regex_pattern, regex=True, na=False)


### Table 1: Commit Distribution Stats
               min  median    max
total_commits  1.0     6.0  693.0

### Global PR Distribution Stats
            min  median    max
unique_prs  1.0     3.0  289.0

### Table 2: Dependency File PR Frequency
                  unique_prs
file_basename               
package.json            7099
pyproject.toml          2180
Cargo.toml              1421
go.mod                  1263
requirements.txt         753
pom.xml                  553
build.gradle             293
composer.json            178
build.gradle.kts         115
Gemfile                   37
Package.swift             13
Pipfile                    7
pubspec.yaml               3


In [91]:
import pandas as pd
import re

# 1. Regex filtering for dependency files
dep_files = [
    "package.json", "requirements.txt", "pyproject.toml", "Pipfile",
    "pom.xml", "build.gradle", "build.gradle.kts", "Cargo.toml",
    "composer.json", "go.mod", "Gemfile", "Package.swift", "pubspec.yaml"
]
regex_pattern = r"(?:^|/)(" + "|".join([re.escape(f) for f in dep_files]) + r")$"

# Filter the dataframe
dep_df = pr_commit_details_df[
    pr_commit_details_df["filename"].str.contains(regex_pattern, regex=True, na=False)
].copy()

# Ensure we have a clean filename for grouping
dep_df['file_type'] = dep_df['filename'].str.split('/').str[-1]

# --- 2. Table: Frequency of Edit Types (Status) ---
status_freq = dep_df['status'].value_counts().reset_index()
status_freq.columns = ['Edit Type', 'Frequency']

# --- 3. Table: Additions, Deletions, and Code Churn per File Type ---
churn_summary = dep_df.groupby('file_type').agg(
    total_additions=('additions', 'sum'),
    total_deletions=('deletions', 'sum'),
    total_churn=('changes', 'sum'),
    count_of_edits=('sha', 'count')
).sort_values(by='total_churn', ascending=False)

# --- 4. Global Totals ---
global_totals = dep_df[['additions', 'deletions', 'changes']].sum().to_frame().T
global_totals.columns = ['Global Additions', 'Global Deletions', 'Global Code Churn']

print("### Edit Type Frequency")
print(status_freq.to_string(index=False))

print("\n### Churn Metrics per Dependency File")
print(churn_summary)

print("\n### Overall Aggregated Totals")
print(global_totals.to_string(index=False))

/var/folders/p4/wn9d12jj6g78td0tl7gx7xlw0000gp/T/ipykernel_59551/2051315619.py:14: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  pr_commit_details_df["filename"].str.contains(regex_pattern, regex=True, na=False)


### Edit Type Frequency
Edit Type  Frequency
 modified      98001
    added       5559
  removed       1736
  renamed        674

### Churn Metrics per Dependency File
                  total_additions  total_deletions  total_churn  \
file_type                                                         
package.json               335603           251964       587567   
go.mod                      86112            74709       160821   
pyproject.toml              69304            19771        89075   
Cargo.toml                  48584            32555        81139   
pom.xml                     45305            32223        77528   
requirements.txt            34850            33997        68847   
build.gradle                19302             8504        27806   
build.gradle.kts             7916             4863        12779   
composer.json                2837             2085         4922   
Package.swift                 480              369          849   
Gemfile                     